# 🍽️ Top Cuisines Analysis
### Restaurant Dataset — Cuisine Frequency & Distribution Study

---

**Objective:** Identify the top three most common cuisines across all restaurants in the dataset and calculate the percentage of restaurants serving each of those cuisines.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

print('✅ Libraries imported successfully.')

## 2. Load & Inspect the Dataset

In [ ]:
df = pd.read_csv('Dataset___2_.csv')

print(f'Dataset Shape : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns       : {list(df.columns)}')
df.head()

## 3. Data Quality Check — Cuisines Column

In [ ]:
total_records = len(df)
null_cuisine  = df['Cuisines'].isna().sum()
valid_records = total_records - null_cuisine

print(f'Total records         : {total_records:,}')
print(f'Missing Cuisine values: {null_cuisine}')
print(f'Valid records used    : {valid_records:,}')

# Drop rows with missing cuisine
df_clean = df.dropna(subset=['Cuisines']).copy()
print('\n✅ Data cleaned — null cuisine rows removed.')

## 4. Cuisine Extraction & Frequency Count

> Each restaurant may serve **multiple cuisines** (comma-separated). We split and explode each entry to count individual cuisine appearances.

In [ ]:
# Split multi-cuisine entries, strip whitespace, and explode into individual rows
all_cuisines = (
    df_clean['Cuisines']
    .str.split(',')
    .explode()
    .str.strip()
)

cuisine_counts = all_cuisines.value_counts()

print(f'Unique cuisines identified: {cuisine_counts.shape[0]}')
print('\nTop 10 Cuisines by Frequency:')
print(cuisine_counts.head(10).to_string())

## 5. Top 3 Cuisines — Count & Percentage

In [ ]:
top3 = cuisine_counts.head(3)

results = pd.DataFrame({
    'Cuisine'           : top3.index,
    'Restaurant Count'  : top3.values,
    'Percentage (%)'    : ((top3.values / valid_records) * 100).round(2)
})
results.index = range(1, 4)
results.index.name = 'Rank'

print('━' * 55)
print(f"{'Rank':<6}{'Cuisine':<18}{'Count':>12}{'Percentage':>14}")
print('━' * 55)
for rank, row in results.iterrows():
    print(f"{rank:<6}{row['Cuisine']:<18}{int(row['Restaurant Count']):>12,}{row['Percentage (%)']:>13.2f}%")
print('━' * 55)

results

## 6. Visualisation — Bar Chart

In [ ]:
colors = ['#2E4057', '#048A81', '#54C6EB']

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    results['Cuisine'],
    results['Restaurant Count'],
    color=colors,
    width=0.5,
    edgecolor='white',
    linewidth=1.2,
    zorder=3
)

# Annotate bars with count and percentage
for bar, pct in zip(bars, results['Percentage (%)']):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 40,
        f'{int(height):,}\n({pct}%)',
        ha='center', va='bottom',
        fontsize=11, fontweight='bold', color='#222222'
    )

ax.set_title('Top 3 Most Common Cuisines in the Restaurant Dataset',
             fontsize=14, fontweight='bold', pad=16, color='#1a1a2e')
ax.set_xlabel('Cuisine Type', fontsize=12, labelpad=8)
ax.set_ylabel('Number of Restaurants', fontsize=12, labelpad=8)
ax.set_ylim(0, max(results['Restaurant Count']) * 1.18)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='y', linestyle='--', alpha=0.6, zorder=0)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('top_cuisines_bar.png', dpi=180, bbox_inches='tight')
plt.show()
print('✅ Bar chart saved as top_cuisines_bar.png')

## 7. Visualisation — Pie / Donut Chart

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

wedges, texts, autotexts = ax.pie(
    results['Restaurant Count'],
    labels=results['Cuisine'],
    autopct='%1.2f%%',
    colors=colors,
    startangle=140,
    pctdistance=0.78,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2)
)

for text in texts:
    text.set_fontsize(12)
    text.set_fontweight('bold')
for autotext in autotexts:
    autotext.set_fontsize(10)
    autotext.set_color('white')
    autotext.set_fontweight('bold')

ax.set_title('Cuisine Share Among Top 3\n(% of Total Valid Restaurants)',
             fontsize=13, fontweight='bold', color='#1a1a2e', pad=20)

# Centre annotation
ax.text(0, 0, f'{valid_records:,}\nRestaurants', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#333333')

plt.tight_layout()
plt.savefig('top_cuisines_donut.png', dpi=180, bbox_inches='tight')
plt.show()
print('✅ Donut chart saved as top_cuisines_donut.png')

## 8. Export Results to CSV

In [ ]:
results.to_csv('top_cuisines_results.csv')
print('✅ Results exported to top_cuisines_results.csv')
print()
print('📊 Final Summary')
print('=' * 55)
for _, row in results.iterrows():
    print(f"  #{_}  {row['Cuisine']:<16} → {int(row['Restaurant Count']):>5,} restaurants  ({row['Percentage (%)']:.2f}%)")
print('=' * 55)

---
## ✅ Task Complete

| Rank | Cuisine | Restaurant Count | Percentage |
|------|---------|-----------------|------------|
| 1 | North Indian | 3,960 | 41.50% |
| 2 | Chinese | 2,735 | 28.66% |
| 3 | Fast Food | 1,986 | 20.81% |

> **Note:** Percentages are calculated with respect to the total number of valid (non-null) restaurant records (9,542). Since a single restaurant may serve multiple cuisines, the percentages can sum beyond 100%.

---